# ISDS Option A — Full Reproducible Workflow
## Prediction of 30-day mortality after acute myocardial infarction

This notebook is the readable, step-by-step companion to `src/isds_option_a_pipeline.py`. It follows the **current** project specification: auditable cleaning, leakage-safe preprocessing, repeated nested cross-validation, candidate-model comparison, class-imbalance ablation, calibration, bootstrap uncertainty, sensitivity analyses, threshold trade-offs, decision-curve analysis and held-out permutation importance.

**Scope:** academic/internal validation only; not a clinical deployment model.

## 1. Setup

Run from the repository root after `pip install -r requirements.txt`. The full implementation is kept in a reusable module so the notebook and command-line workflow use the same code.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path('src').resolve()))
import isds_option_a_pipeline as isds

DATA = Path('ami_patient_data.csv')
OUTPUT = Path('outputs/notebook_run')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Seed:', isds.RANDOM_SEED)

## 2. Raw-data audit and deterministic cleaning

The analysis does not silently overwrite the raw source. It renames two misspelled headers, converts encoded unknown/invalid values to missing, and corrects two unmistakable height unit-entry errors.

In [ ]:
raw, data = isds.load_and_clean(DATA)
audit = isds.audit_table(raw, data)
display(audit)
display(data.isna().sum().sort_values(ascending=False).to_frame('missing_cells').head(10))

## 3. Exploratory data analysis

The first diagnostics quantify the rare 30-day mortality outcome and the pattern of missingness after auditable recoding.

In [ ]:
fig_dir = OUTPUT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
isds.plot_eda(data, fig_dir)
print('Saved EDA figures to', fig_dir)

## 4. Leakage-safe preprocessing

Continuous/ordinal variables use fold-specific median imputation and standardisation. Binary variables use fold-specific most-frequent imputation. Smoking is imputed and one-hot encoded. These transformations are fitted only on training data within validation.

In [ ]:
X = data.drop(columns=[isds.OUTCOME])
y = data[isds.OUTCOME].astype(int)
preprocessor = isds.make_preprocessor()
print('X shape:', X.shape, '| deaths:', int(y.sum()), '| event rate:', f'{y.mean():.2%}')

## 5. Repeated nested internal validation

Outer validation: 5 stratified folds repeated 5 times. Inner tuning: 4 stratified folds. The tuning objective is log loss because the task requires reliable risk probabilities, not only ranking.

In [ ]:
models = isds.build_models(preprocessor)
ridge_model, ridge_grid = models['Ridge logistic']
ridge_rep, ridge_oof, ridge_tuning = isds.nested_oof(ridge_model, ridge_grid, X, y)
display(pd.DataFrame([isds.metrics(ridge_oof)]))
display(ridge_tuning.head())

## 6. Candidate-model comparison

Compare ridge, standard logistic regression, Random Forest and Gradient Boosting under the same held-out validation design. An intercept-only model provides a prevalence reference.

In [ ]:
candidate_names = ['Ridge logistic', 'Standard logistic', 'Random Forest', 'Gradient Boosting']
results = []
oof = {}
for name in candidate_names:
    model, grid = models[name]
    rep, patient, tuning = isds.nested_oof(model, grid, X, y)
    oof[name] = patient
    results.append({'model': name, **isds.metrics(patient)})

_, null_patient = isds.null_oof(X, y)
results.append({'model': 'Intercept-only reference', **isds.metrics(null_patient)})
display(pd.DataFrame(results))

## 7. Class-imbalance ablation

Mortality is rare (6.62%), but rebalancing is retained only if it improves held-out prediction. Compare matched ridge models with no rebalancing, class weighting and random oversampling.

In [ ]:
imbalance_rows = []
for name in ['Ridge logistic', 'Ridge class-weighted', 'Ridge oversampled']:
    model, grid = models[name]
    _, patient, _ = isds.nested_oof(model, grid, X, y)
    imbalance_rows.append({'strategy': name, **isds.metrics(patient)})
display(pd.DataFrame(imbalance_rows))

## 8. Calibration and bootstrap uncertainty

Calibration intercept/slope are estimated from genuine out-of-fold ridge probabilities. Headline metric uncertainty uses 2,000 patient-level bootstrap resamples of those predictions.

In [ ]:
display(pd.DataFrame([isds.calibration_summary(ridge_oof)]))
display(isds.bootstrap_intervals(ridge_oof, n_boot=2000))

## 9. Sensitivity analyses

Two robustness checks alter only one modelling choice at a time: categorical Killip coding and explicit missingness indicators.

In [ ]:
ridge_grid = {'model__C': [0.01, 0.1, 1.0]}

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

killip_model = Pipeline([
    ('preprocessor', isds.make_killip_categorical_preprocessor()),
    ('model', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=20000, tol=1e-4, random_state=isds.RANDOM_SEED)),
])
_, killip_oof, _ = isds.nested_oof(killip_model, ridge_grid, X, y)

mi_model = Pipeline([
    ('preprocessor', isds.make_missing_indicator_preprocessor()),
    ('model', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=20000, tol=1e-4, random_state=isds.RANDOM_SEED)),
])
_, mi_oof, _ = isds.nested_oof(mi_model, ridge_grid, X, y)

display(pd.DataFrame([
    {'specification': 'Primary ridge', **isds.metrics(ridge_oof)},
    {'specification': 'Killip categorical', **isds.metrics(killip_oof)},
    {'specification': 'Missing indicators added', **isds.metrics(mi_oof)},
]))

## 10. Threshold trade-offs and decision-curve analysis

The 5%, 10%, 15% and 20% thresholds are illustrative rather than optimized clinical cut-offs. Decision curves compare net benefit with treat-all and treat-none strategies.

In [ ]:
import numpy as np
y_oof = ridge_oof['y_true'].to_numpy()
p_oof = ridge_oof['predicted_probability'].to_numpy()
threshold_table = pd.DataFrame([isds.threshold_metrics(y_oof, p_oof, t) for t in [0.05, 0.10, 0.15, 0.20]])
display(threshold_table)

dca = isds.decision_curve(y_oof, p_oof, np.linspace(0.01, 0.30, 60))
display(dca.head())

## 11. Final model coefficients

The full-data ridge fit uses `C = 0.1`, the modal tuned value. Coefficients are descriptive predictive associations. Standardized continuous/ordinal predictors have odds ratios corresponding approximately to a one-standard-deviation increase.

In [ ]:
final_model, coefficients = isds.final_coefficients(X, y, preprocessor)
display(coefficients.head(15))

## 12. Held-out permutation importance

Each raw predictor is shuffled repeatedly in outer holdout folds while the fold-specific ridge model remains fixed. Importance is the increase in held-out log loss.

In [ ]:
importance = isds.heldout_raw_permutation_importance(X, y, preprocessor)
display(importance.head(15))

## 13. One-command reproducibility

For a complete run that saves all tables, figures, metadata and the final serialized pipeline, use the script entry point below. This repeats the full nested-validation workflow and can take time on a CPU.

In [ ]:
# Uncomment to run the full export pipeline.
# isds.run(DATA, Path('outputs/current'))

## Interpretation boundary

The final model is internally validated only. It does not establish causal effects, treatment benefit, or transportability to other hospitals/populations. External validation would be required before clinical use.